In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

d:\ProteinPrediction\MainFiles\GPU_VENV\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


In [3]:
import numpy as np
import torch

# load numpy arrays
X_train_w = np.load("../data/Saved_Windows_features/X_train.npy")
y_train_w = np.load("../data/Saved_Windows_features/y_train.npy")

X_val_w = np.load("../data/Saved_Windows_features/X_val.npy")
y_val_w = np.load("../data/Saved_Windows_features/y_val.npy")

X_test_w = np.load("../data/Saved_Windows_features/X_test.npy")
y_test_w = np.load("../data/Saved_Windows_features/y_test.npy")

Reshaping the feature vector to ( N, 11, 41 )

In [4]:
X_train_seq = X_train_w.reshape(-1, 11, 41)
X_val_seq = X_val_w.reshape(-1, 11, 41)
X_test_seq = X_test_w.reshape(-1, 11, 41)

print(X_train_seq.shape)

(3337092, 11, 41)


In [5]:
X_train_tensor = torch.tensor(X_train_seq, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_w, dtype=torch.float32)

X_val_tensor = torch.tensor(X_val_seq, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_w, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test_seq, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_w, dtype=torch.float32)

Dataset Loader

In [6]:
class ProteinDataset(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

DataLoaders

In [7]:
batch_size = 4096

train_loader = DataLoader(
    ProteinDataset(X_train_tensor, y_train_tensor),
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    ProteinDataset(X_val_tensor, y_val_tensor),
    batch_size=batch_size
)

test_loader = DataLoader(
    ProteinDataset(X_test_tensor, y_test_tensor),
    batch_size=batch_size
)

Training the MLP model 

In [8]:
class MLPRegressor(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(

            nn.Flatten(),

            nn.Linear(11*41, 512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, 4)
        )

    def forward(self, x):
        return self.net(x)

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MLPRegressor().to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [10]:
epochs = 15

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        preds = model(X_batch)

        loss = criterion(preds, y_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} Loss: {train_loss/len(train_loader):.4f}")

Epoch 1/15 Loss: 1221176.3694
Epoch 2/15 Loss: 1000041.1149
Epoch 3/15 Loss: 969548.0378
Epoch 4/15 Loss: 951594.7999
Epoch 5/15 Loss: 937514.9246
Epoch 6/15 Loss: 930702.0606
Epoch 7/15 Loss: 927118.8866
Epoch 8/15 Loss: 924926.5238
Epoch 9/15 Loss: 921893.9302
Epoch 10/15 Loss: 918725.2692
Epoch 11/15 Loss: 916393.2948
Epoch 12/15 Loss: 911880.1815
Epoch 13/15 Loss: 908471.3527
Epoch 14/15 Loss: 905884.8993
Epoch 15/15 Loss: 901263.7132


In [11]:
model.eval()

preds = []
targets = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        preds.append(outputs.cpu().numpy())
        targets.append(y_batch.numpy())

preds = np.vstack(preds)
targets = np.vstack(targets)

In [12]:
rmse = np.sqrt(mean_squared_error(targets, preds))
mae = mean_absolute_error(targets, preds)
r2 = r2_score(targets, preds)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

RMSE: 640.918286523329
MAE: 184.55235290527344
R2: 0.6229566335678101
